In [1]:
!pip install pandas requests pyarrow


In [6]:
import pandas as pd
import os
import glob

RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

TARGET_SYMBOLS = ['GOLDM', 'GOLDGUINEA', 'GOLDPETAL', 'GOLDTEN']

print("--- Running Pipeline on Downloaded Bhavcopy Files ---")

raw_files = glob.glob(os.path.join(RAW_DIR, "*.csv"))
print(f"Found {len(raw_files)} CSV files in {RAW_DIR}\n")

all_gold_records = []

for file_path in raw_files:
    try:
        # Quick validation to skip HTML block pages if any exist
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            first_line = f.readline()
            if '<!doctype html>' in first_line.lower() or '<html' in first_line.lower():
                print(f"Skipped HTML page: {os.path.basename(file_path)}")
                continue
                
            # Find header offset dynamically
            f.seek(0)
            lines = f.readlines()
            
        header_idx = 0
        for idx, line in enumerate(lines[:10]):
            if any(k in line.lower() for k in ['symbol', 'instrument']):
                header_idx = idx
                break

        df = pd.read_csv(file_path, skiprows=header_idx, on_bad_lines='skip')
        df.columns = [c.strip() for c in df.columns]

        # Case-insensitive column search for Symbol
        col_map = {c.lower(): c for c in df.columns}
        sym_col = col_map.get('symbol') or col_map.get('instrumentname')

        if sym_col and sym_col in df.columns:
            df[sym_col] = df[sym_col].astype(str).str.strip()
            gold_df = df[df[sym_col].isin(TARGET_SYMBOLS)].copy()
            
            if not gold_df.empty:
                all_gold_records.append(gold_df)
                print(f"Loaded {len(gold_df)} gold records from: {os.path.basename(file_path)}")
    except Exception as e:
        print(f"Error parsing {os.path.basename(file_path)}: {e}")

if all_gold_records:
    master_df = pd.concat(all_gold_records, ignore_index=True)

    # Save deliverables for Person B
    sample_path = os.path.join(PROCESSED_DIR, "master_gold_contracts_sample.csv")
    parquet_path = os.path.join(PROCESSED_DIR, "master_gold_contracts_12m.parquet")

    master_df.head(200).to_csv(sample_path, index=False)
    master_df.to_parquet(parquet_path, index=False)

    print("\n==========================================")
    print(" PIPELINE COMPLETE & SUCCESSFUL!")
    print(f" Total Gold Records Consolidated: {len(master_df)}")
    print(f" Sample CSV: {sample_path}")
    print(f" Full Dataset: {parquet_path}")
    print("==========================================")
else:
    print("\nNo gold contract records found. Ensure .csv files are placed in data/raw/")

--- Running Pipeline on Downloaded Bhavcopy Files ---
Found 56 CSV files in ../data/raw

Loaded 642 gold records from: BhavCopyDateWise_01042026.csv
Loaded 582 gold records from: BhavCopyDateWise_01072026.csv
Loaded 534 gold records from: BhavCopyDateWise_02062026.csv
Loaded 576 gold records from: BhavCopyDateWise_02092026.csv
Loaded 852 gold records from: BhavCopyDateWise_02122025.csv
Loaded 2294 gold records from: BhavCopyDateWise_03022026.csv
Loaded 1388 gold records from: BhavCopyDateWise_04032026.csv
Loaded 528 gold records from: BhavCopyDateWise_04082026.csv
Loaded 1002 gold records from: BhavCopyDateWise_05012026.csv
Loaded 552 gold records from: BhavCopyDateWise_05052026.csv
Loaded 1210 gold records from: BhavCopyDateWise_06102025.csv
Loaded 1346 gold records from: BhavCopyDateWise_06112025.csv
Loaded 786 gold records from: BhavCopyDateWise_07072026.csv
Loaded 846 gold records from: BhavCopyDateWise_08042026.csv
Loaded 778 gold records from: BhavCopyDateWise_08092026.csv
Loaded